# MBPP Evaluation: Your 100M Python-Code Model

This notebook loads your `nanoGPT`-style 100M model's weights straight from the Hugging Face Hub (no local checkpoint needed) and evaluates it on the **real MBPP benchmark** (Mostly Basic Python Problems) — a standard 500-problem Python coding benchmark — reporting **pass@1**.

**Steps:**
1. Install packages
2. Config — which HF repo to load, MBPP split, generation settings
3. Rebuild the model architecture (identical to your training notebook)
4. Load the pretrained/fine-tuned weights from Hugging Face
5. Load the MBPP dataset
6. Build prompts, generate code, run the real MBPP tests
7. Report pass@1 and save results

⚠️ **Security note:** don't hardcode your HF token in this notebook. Cell 2 reads it from an environment variable or prompts you securely with `getpass` — please rotate any token that was ever committed in plaintext.

## 1. Install packages

In [ ]:
!pip install -q -U datasets transformers huggingface_hub safetensors tqdm


## 2. Config — fill these in

- `HF_REPO`: the Hugging Face repo holding `config.json` + `model.safetensors` — can be your **base** pretrained repo or your **fine-tuned** repo, same file format either way.
- `HF_TOKEN`: only needed if the repo is private. Never hardcode this — the cell below pulls it from the `HF_TOKEN` environment variable, or prompts you securely if it's not set.

In [ ]:
import os
from getpass import getpass

# ======================= FILL THIS IN =======================
HF_REPO = "Ananda100/100m-sftd-python"   # the HF repo to evaluate (base OR fine-tuned)
# ===============================================================

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass("HF token (leave blank if the repo is public): ") or None

MBPP_SPLIT = "test"          # "test" = the standard 500-problem MBPP eval set
NUM_PROBLEMS = None          # set to a small int (e.g. 20) for a quick smoke test, None = full split

MAX_NEW_TOKENS = 256
TEMPERATURE = 0.2
TOP_K = 50
TIMEOUT_S = 5                # seconds allowed per problem's test execution

OUT_FILE = "mbpp_results.json"


## 3. Model architecture

Identical to your training notebook (`base100m.ipynb` / the SFT notebook) — required to load the state dict correctly. If you ever change the architecture during training, this cell must match.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(torch.nn.functional, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 32256
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        dev = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"sequence length {t} exceeds block_size {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=dev)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_token_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            if eos_token_id is not None and idx.size(0) == 1 and idx_next.item() == eos_token_id:
                break
        return idx


## 4. Load the pretrained/fine-tuned weights from Hugging Face

Downloads `config.json` + `model.safetensors` from `HF_REPO` and builds the model directly from the Hub — no local checkpoint file required.

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import json as _json

device = "cuda" if torch.cuda.is_available() else "cpu"

config_path = hf_hub_download(repo_id=HF_REPO, filename="config.json", token=HF_TOKEN)
weights_path = hf_hub_download(repo_id=HF_REPO, filename="model.safetensors", token=HF_TOKEN)

with open(config_path) as f:
    cfg_dict = _json.load(f)

config = GPTConfig(**cfg_dict)
model = GPT(config)

state_dict = load_file(weights_path)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Loaded {HF_REPO} — {n_params:.2f}M params on {device}")


## 5. Load the tokenizer

Same `deepseek-ai/deepseek-coder-6.7b-base` tokenizer used for pretraining/fine-tuning.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base", trust_remote_code=True)
if tokenizer.eos_token_id is None:
    raise ValueError("Tokenizer has no eos_token_id set - fix before generating.")
print(f"Vocab size: {len(tokenizer)}")


## 6. Load the MBPP dataset

Pulls the real [MBPP benchmark](https://huggingface.co/datasets/google-research-datasets/mbpp) from the Hub. The `test` split is the standard 500-problem evaluation set.

In [ ]:
from datasets import load_dataset

print(f"Loading MBPP ({MBPP_SPLIT} split)...")
mbpp = load_dataset("google-research-datasets/mbpp", "full", split=MBPP_SPLIT)
if NUM_PROBLEMS:
    mbpp = mbpp.select(range(min(NUM_PROBLEMS, len(mbpp))))
print(f"Evaluating on {len(mbpp)} MBPP problems")
mbpp[0]


## 7. Prompt construction

MBPP gives a plain-English `text` description but **no function signature**, so a small model has almost no chance of guessing the exact expected function name. As standard MBPP eval harnesses do, we show the model one test assertion up front so it knows the expected name/signature — then build the same `### Problem / ### Solution` template your model was trained on.

In [ ]:
def build_prompt(example):
    text = example["text"]
    first_test = example["test_list"][0] if example["test_list"] else ""
    problem = f"{text}\nYour code should satisfy this test:\n{first_test}"
    return f"### Problem\n{problem}\n\n### Solution\n```python\n"

def extract_code(generated_text, prompt):
    completion = generated_text[len(prompt):]
    if "```" in completion:
        completion = completion.split("```")[0]
    return completion

print(build_prompt(mbpp[0]))


## 8. Sandboxed test execution

Runs the candidate code plus MBPP's real `assert`-based `test_list` (and any `test_setup_code`) with a timeout, mirroring your training notebook's evaluation cell.

In [ ]:
import signal
import multiprocessing as mp
from contextlib import contextmanager

class GenTimeout(Exception):
    pass

def _handler(signum, frame):
    raise GenTimeout()

@contextmanager
def time_limit(seconds):
    # Still fine for wrapping our OWN trusted code (e.g. model.generate).
    # NOT safe for untrusted/generated code -- see run_mbpp_tests below.
    signal.signal(signal.SIGALRM, _handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)

def _mbpp_test_worker(code, test_setup_code, test_list, result_queue):
    """Runs in a child process so it can be killed unconditionally,
    even if the generated code has a bare `except:` that would otherwise
    swallow a signal-based timeout."""
    namespace = {}
    try:
        if test_setup_code:
            exec(test_setup_code, namespace)
        exec(code, namespace)
        passed, total = 0, len(test_list)
        for t in test_list:
            try:
                exec(t, namespace)
                passed += 1
            except Exception:
                pass
        result_queue.put((passed, total))
    except Exception:
        result_queue.put((0, len(test_list)))

def run_mbpp_tests(code, test_setup_code, test_list, timeout_s=5):
    ctx = mp.get_context("fork")  # fork is fine on Colab/Linux
    result_queue = ctx.Queue()
    proc = ctx.Process(
        target=_mbpp_test_worker,
        args=(code, test_setup_code, test_list, result_queue),
        daemon=True,
    )
    proc.start()
    proc.join(timeout_s)

    if proc.is_alive():
        # Candidate code is hung (or swallowing signals internally) -- kill it
        # from the outside. This works no matter what the code inside does.
        proc.terminate()
        proc.join(1)
        if proc.is_alive():
            proc.kill()
            proc.join()
        return 0, len(test_list)

    try:
        return result_queue.get_nowait()
    except Exception:
        return 0, len(test_list)


## 9. Generate + evaluate

Generates a completion for every MBPP problem and scores it against the real tests. This is the slow cell — progress bar included.

In [ ]:
from tqdm.auto import tqdm
import time

GEN_TIMEOUT_S = TIMEOUT_S * 4  # generation gets a bigger budget than test execution

results = []
for ex in tqdm(mbpp, desc="MBPP eval"):
    prompt = build_prompt(ex)
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    idx = torch.tensor([ids], dtype=torch.long, device=device)

    t0 = time.time()
    try:
        with time_limit(GEN_TIMEOUT_S):
            with torch.no_grad():
                out = model.generate(
                    idx,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=TEMPERATURE,
                    top_k=TOP_K,
                    eos_token_id=tokenizer.eos_token_id,
                )
        generated = tokenizer.decode(out[0].tolist())
        code_str = extract_code(generated, prompt)
        passed, total = run_mbpp_tests(code_str, ex.get("test_setup_code", ""), ex["test_list"], timeout_s=TIMEOUT_S)
    except GenTimeout:
        print(f"[timeout] task_id={ex['task_id']} generation exceeded {GEN_TIMEOUT_S}s, scoring as failed")
        passed, total = 0, len(ex["test_list"])

    elapsed = time.time() - t0
    if elapsed > GEN_TIMEOUT_S * 0.5:
        print(f"[slow] task_id={ex['task_id']} took {elapsed:.1f}s")

    results.append({
        "task_id": ex["task_id"],
        "passed": passed,
        "total": total,
        "full_pass": passed == total and total > 0,
    })

print(f"Done: {len(results)} problems evaluated")


## 10. Report pass@1 and save results

In [ ]:
import json

if not results:
    print("No MBPP problems evaluated.")
else:
    pass_at_1 = sum(r["full_pass"] for r in results) / len(results)
    avg_test_pass_rate = sum(r["passed"] / r["total"] for r in results if r["total"] > 0) / len(results)

    print("--- MBPP results ---")
    print(f"Repo:                {HF_REPO}")
    print(f"Problems evaluated:  {len(results)}")
    print(f"pass@1 (all tests pass): {pass_at_1:.1%}")
    print(f"Average per-test pass rate: {avg_test_pass_rate:.1%}")

    with open(OUT_FILE, "w") as f:
        json.dump({
            "repo": HF_REPO,
            "split": MBPP_SPLIT,
            "num_problems": len(results),
            "pass_at_1": pass_at_1,
            "avg_test_pass_rate": avg_test_pass_rate,
            "per_problem": results,
        }, f, indent=2)
    print(f"Saved detailed results to {OUT_FILE}")


## Notes

- A ~100M-parameter model with no MBPP-specific fine-tuning typically scores low single digits to low teens % pass@1 — that's expected, not a sign something's broken.
- `HF_REPO` can point at either your base pretrained repo or your fine-tuned repo; both use the same `config.json` + `model.safetensors` format, so nothing else in this notebook needs to change.
- If you want a quick sanity check before running the full 500-problem split, set `NUM_PROBLEMS = 20` in the config cell.